# 02 Feature Engineering

            This notebook converts the cleaned bank marketing data into model-ready matrices. Binary yes/no variables are label-encoded, categorical predictors are one-hot encoded with the first level dropped, and numeric variables are kept in their original scale.


## Define Input and Output Paths

            Paths are created for both modeling scenarios: the production-safe no-duration dataset and the diagnostic with-duration dataset. The production preprocessor is saved as `preprocessor.pkl`, which is the artifact used by the API.


In [1]:
# Define cleaned-data, feature-matrix, and preprocessor artifact paths.

from pathlib import Path
import sys
import pandas as pd
import logging

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

CLEANED_NO_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data.csv'
CLEANED_WITH_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data_with_duration.csv'
FEATURED_NO_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'featured_bank_data.csv'
FEATURED_WITH_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'featured_bank_data_with_duration.csv'
PREPROCESSOR = PROJECT_ROOT / 'models' / 'trained' / 'preprocessor.pkl'
PREPROCESSOR_WITH_DURATION = PROJECT_ROOT / 'models' / 'trained' / 'preprocessor_with_duration.pkl'


## Apply Feature Rules

            This preview shows the direct feature rules before one-hot encoding. `default`, `housing`, and `loan` are converted from `yes/no` strings into `1/0` indicators.


In [2]:
# Preview the label-encoding step before one-hot encoding.

from src.features.engineer import create_features

cleaned = pd.read_csv(CLEANED_NO_DURATION)
create_features(cleaned).head()


2026-05-17 16:16:14,615 - bank-feature-engineering - INFO - Creating bank marketing features


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,previous,poutcome,deposit
0,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1,0,unknown,1
1,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1,0,unknown,1
2,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1,0,unknown,1
3,55,services,married,secondary,0,2476,1,0,unknown,5,may,1,0,unknown,1
4,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,2,0,unknown,1


The target is already numeric (`deposit = 1` for yes, `0` for no). Categorical variables such as `job`, `marital`, `education`, `contact`, `month`, and `poutcome` remain as text so the preprocessor can one-hot encode them consistently.


## Build the Production Feature Matrix

            The preprocessor is fit on the no-duration dataset and saved for later inference. Full-rank one-hot encoding is used by dropping the first category for each categorical feature.


In [3]:
# Fit full-rank one-hot encoding and save the production preprocessor.

from src.features.engineer import run_feature_engineering

logging.disable(logging.INFO)   # suppress INFO logs

model_ready = run_feature_engineering(CLEANED_NO_DURATION, FEATURED_NO_DURATION, PREPROCESSOR)

logging.disable(logging.NOTSET)  # re-enable logging

model_ready.head()


,age,balance,day,campaign,previous,default,housing,loan,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,deposit
0,59.0,2343.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,56.0,45.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
2,41.0,1270.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
3,55.0,2476.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
4,54.0,184.0,5.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1


The resulting production matrix is used by the training script. Since `duration` is excluded, this feature set is suitable for customer scoring before campaign outreach.


## Confirm Feature Matrix Shape

            The matrix shape and first feature names are checked to confirm that preprocessing generated the expected numeric and dummy-encoded columns.


In [4]:
# Confirm output shape and inspect the first generated feature names.

model_ready.shape, model_ready.columns[:12].tolist()


((11162, 41),
 ['age',
  'balance',
  'day',
  'campaign',
  'previous',
  'default',
  'housing',
  'loan',
  'job_blue-collar',
  'job_entrepreneur',
  'job_housemaid',
  'job_management'])

The production feature matrix has 11,162 rows and 41 columns, including the target. This output is compact, reproducible, and directly compatible with the model-training pipeline.


## Build the With-Duration Comparison Matrix

            A separate feature matrix is created with `duration` retained. This matrix is used only to measure how much model performance changes when post-call information is included.


In [5]:
# Build the diagnostic feature matrix that includes duration.

logging.disable(logging.INFO)   # suppress INFO logs

model_ready_duration = run_feature_engineering(CLEANED_WITH_DURATION, FEATURED_WITH_DURATION, PREPROCESSOR_WITH_DURATION)
logging.disable(logging.NOTSET)  # re-enable logging

model_ready_duration.shape


(11162, 42)

The with-duration matrix has 42 columns, exactly one more than the production matrix. This keeps the comparison controlled because the only added feature is `duration`.
